# Run the Dialogflow ES latency experiments (Google Colab)

This notebook reproduces the measurements in *Performance Characterization of Dialogflow ES as Conversational Middleware*. It runs the harness in `harness/bench_harness.py` and writes raw logs that `analysis/analysis.py` turns into the paper's tables and figures.

**You need:**
1. Four Dialogflow ES agents: restore the ZIPs in `agents/` (Dialogflow console → ⚙ → Export and Import → Restore from ZIP).
2. A service-account key (JSON) for each agent's Google Cloud project, with the *Dialogflow API Client* role.
3. An ngrok account (authtoken) and an OpenWeatherMap API key.
4. Colab Secrets (🔑 in the left sidebar): `NGROK_AUTHTOKEN` and `OPENWEATHER_KEY`, with *Notebook access* switched on.

**Run the cells one at a time.** Logs are written to Google Drive, so a disconnect does not lose data.

In [ ]:
# 1. Get the code and install dependencies
!git clone https://github.com/E-Modiri/dialogflow-latency-study.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# 2. Save logs to Google Drive (survives disconnects)
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/dialogflow_latency_bench_out"

### 3. Configuration
Upload the four service-account key files (folder icon → upload) and fill in the project IDs and file names.

In [ ]:
import os, sys
from google.colab import userdata
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
os.environ["OPENWEATHER_KEY"] = userdata.get("OPENWEATHER_KEY")

os.environ["DF_PROJECT_WEATHER"]  = "YOUR-WEATHER-PROJECT-ID"
os.environ["DF_PROJECT_CAPITAL"]  = "YOUR-CAPITAL-PROJECT-ID"
os.environ["DF_PROJECT_CURRENCY"] = "YOUR-CURRENCY-PROJECT-ID"
os.environ["DF_PROJECT_JOKE"]     = "YOUR-JOKE-PROJECT-ID"
os.environ["DF_CREDENTIALS_WEATHER"]  = "/content/weather-key.json"
os.environ["DF_CREDENTIALS_CAPITAL"]  = "/content/capital-key.json"
os.environ["DF_CREDENTIALS_CURRENCY"] = "/content/currency-key.json"
os.environ["DF_CREDENTIALS_JOKE"]     = "/content/joke-key.json"
for k in ("WEATHER", "CAPITAL", "CURRENCY", "JOKE"):
    f = os.environ[f"DF_CREDENTIALS_{k}"]
    print(f"{k:9s} key file {'found' if os.path.exists(f) else 'MISSING'}: {f}")

sys.path.insert(0, "/content/repo/harness")
import bench_harness as H
cfg = H.Config(
    out_dir=OUT_DIR,
    queries_file="/content/repo/data/queries.json",
    intents={"weather": "How is the weather?", "capital": "GetCapital",
             "currency": "ConvertCurrency", "joke": "GetJoke"},
    burst_sizes=(10, 20, 40, 60, 80, 90),
    sustained_rates_rps=(0.5, 1.0, 2.0, 3.0),
)
H.init(cfg)   # starts the webhook and ngrok tunnel; prints one fulfillment URL per domain

### 4. Point each agent at the webhook
In each agent: **Fulfillment → Webhook → URL** = the printed URL for that domain (ending `/webhook/weather`, `/webhook/capital`, `/webhook/currency` or `/webhook/joke`) → **Save**. The URL changes whenever the runtime restarts.

In [ ]:
# 5. Preflight: one real request per configuration and domain. Must print PREFLIGHT PASSED.
qs = H.load_queries()
H.preflight()

In [ ]:
# 6. Optional: check Currency entity extraction against ground truth (Dialogflow only, ~2 min)
import time, pandas as pd
from google.cloud import dialogflow_v2 as dialogflow
from google.protobuf.json_format import MessageToDict
client, project = H.df_client("currency"), H.CFG.df_projects["currency"]
def code_of(x):
    x = x[0] if isinstance(x, list) and x else x
    x = x.get("currency", "") if isinstance(x, dict) else x
    return str(x or "").strip().upper()
rows = []
for lbl, txt in qs["currency"]:
    s = client.session_path(project, f"diag-{lbl}-{int(time.time())}")
    r = client.detect_intent(request={"session": s, "query_input": dialogflow.QueryInput(
        text=dialogflow.TextInput(text=txt, language_code="en"))})
    p = MessageToDict(r._pb)["queryResult"].get("parameters", {})
    f, t = code_of(p.get("from_currency")), code_of(p.get("to_currency"))
    rows.append({"pair": lbl, "query": txt, "from": f, "to": t, "correct": (f, t) == tuple(lbl.split("_"))})
    time.sleep(1)
diag = pd.DataFrame(rows)
print(f"Correct: {diag.correct.sum()} / {len(diag)}")
diag.to_csv(f"{OUT_DIR}/currency_extraction_check.csv", index=False)

### 7. Experiments: one domain per cell (about 70 minutes each)
Each call runs the sequential, burst and sustained tests for one domain. Keep the tab visible and the computer awake.

In [ ]:
H.run_all(domains=("weather",))

In [ ]:
H.run_all(domains=("capital",))

In [ ]:
H.run_all(domains=("currency",))

In [ ]:
H.preflight(("joke",))
H.run_sequential("joke", qs["joke"], reps=30)   # 30 blocks: repeated measurements per category
H.run_bursts("joke", qs["joke"])
H.run_sustained("joke", qs["joke"])
H.write_metadata({"public_webhook": "redacted"})

In [ ]:
# 8. Analysis: tables, figures and numbers.json from the raw logs
!python /content/repo/analysis/analysis.py "{OUT_DIR}/attempts.csv" --out /content/results
import shutil; shutil.make_archive("/content/results", "zip", "/content/results")
from google.colab import files; files.download("/content/results.zip")